In [7]:
import pandas as pd

order_products = pd.read_csv(
    "../data/raw/order_products__prior.csv",
    usecols=["order_id", "product_id"],
    nrows=200000   
)


products = pd.read_csv(
    "../data/raw/products.csv",
    usecols=["product_id", "product_name"]
)

print("order_products shape:", order_products.shape)
print("products shape:", products.shape)

order_products shape: (200000, 2)
products shape: (49688, 2)


In [8]:
merged_df = order_products.merge(products, on="product_id", how="left")



In [9]:
transactions = merged_df.groupby("order_id")["product_name"].apply(list)
transactions_list = transactions.tolist()

transactions.head()
transactions_list[:5]

[['Organic Egg Whites',
  'Michigan Organic Kale',
  'Garlic Powder',
  'Coconut Butter',
  'Natural Sweetener',
  'Carrots',
  'Original Unflavored Gelatine Mix',
  'All Natural No Stir Creamy Almond Butter',
  'Classic Blend Cole Slaw'],
 ['Total 2% with Strawberry Lowfat Greek Strained Yogurt',
  'Unsweetened Almondmilk',
  'Lemons',
  'Organic Baby Spinach',
  'Unsweetened Chocolate Almond Breeze Almond Milk',
  'Organic Ginger Root',
  'Air Chilled Organic Boneless Skinless Chicken Breasts',
  'Organic Ezekiel 49 Bread Cinnamon Raisin'],
 ['Plain Pre-Sliced Bagels',
  'Honey/Lemon Cough Drops',
  'Chewy 25% Low Sugar Chocolate Chip Granola',
  'Oats & Chocolate Chewy Bars',
  "Kellogg's Nutri-Grain Apple Cinnamon Cereal",
  'Nutri-Grain Soft Baked Strawberry Cereal Breakfast Bars',
  "Kellogg's Nutri-Grain Blueberry Cereal",
  'Tiny Twists Pretzels',
  'Traditional Snack Mix',
  'Goldfish Cheddar Baked Snack Crackers',
  'Original Orange Juice',
  'Sugarfree Energy Drink',
  'Ener

In [10]:
import mlxtend
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions_list).transform(transactions_list)

basket_df = pd.DataFrame(te_array, columns=te.columns_)

In [11]:
item_support = basket_df.sum() / len(basket_df)
frequent_items = item_support[item_support > 0.01].index
basket_df = basket_df[frequent_items]

basket_df.shape

(19850, 104)

In [12]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    basket_df,
    min_support=0.005,
    use_colnames=True
)

frequent_itemsets.head()

,support,itemsets
0,0.012191,frozenset({100% Raw Coconut Water})
1,0.019194,frozenset({100% Whole Wheat Bread})
2,0.012897,frozenset({2% Reduced Fat Milk})
3,0.025945,frozenset({Apple Honeycrisp Organic})
4,0.019093,frozenset({Asparagus})


In [13]:
frequent_itemsets.sort_values(by="support", ascending=False).head(10)

,support,itemsets
6,0.149622,frozenset({Banana})
5,0.119698,frozenset({Bag of Organic Bananas})
76,0.078338,frozenset({Organic Strawberries})
38,0.073904,frozenset({Organic Baby Spinach})
58,0.066650,frozenset({Organic Hass Avocado})
34,0.055516,frozenset({Organic Avocado})
29,0.045995,frozenset({Large Lemon})
97,0.045340,frozenset({Strawberries})
67,0.043174,frozenset({Organic Raspberries})
31,0.042922,frozenset({Limes})


In [14]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules.head()
rules.sort_values(by="lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
113,frozenset({Organic Cilantro}),frozenset({Limes}),0.020252,0.042922,0.005793,0.286070,6.664886,1.0,0.004924,1.340576,0.867529,0.100966,0.254052,0.210523
112,frozenset({Limes}),frozenset({Organic Cilantro}),0.042922,0.020252,0.005793,0.134977,6.664886,1.0,0.004924,1.132626,0.888078,0.100966,0.117096,0.210523
140,frozenset({Organic Garlic}),frozenset({Organic Yellow Onion}),0.033048,0.034408,0.006599,0.199695,5.803731,1.0,0.005462,1.206530,0.855986,0.108444,0.171177,0.195748
141,frozenset({Organic Yellow Onion}),frozenset({Organic Garlic}),0.034408,0.033048,0.006599,0.191801,5.803731,1.0,0.005462,1.196428,0.857191,0.108444,0.164179,0.195748
101,frozenset({Large Lemon}),frozenset({Limes}),0.045995,0.042922,0.008262,0.179628,4.184986,1.0,0.006288,1.166639,0.797743,0.102436,0.142837,0.186058
100,frozenset({Limes}),frozenset({Large Lemon}),0.042922,0.045995,0.008262,0.192488,4.184986,1.0,0.006288,1.181413,0.795181,0.102436,0.153556,0.186058
142,frozenset({Organic Hass Avocado}),frozenset({Organic Lemon}),0.066650,0.029219,0.007809,0.117158,4.009631,1.0,0.005861,1.099609,0.804200,0.088673,0.090586,0.192200
143,frozenset({Organic Lemon}),frozenset({Organic Hass Avocado}),0.029219,0.066650,0.007809,0.267241,4.009631,1.0,0.005861,1.273748,0.773192,0.088673,0.214916,0.192200
137,frozenset({Organic Hass Avocado}),frozenset({Organic Cucumber}),0.066650,0.024181,0.005441,0.081633,3.375850,1.0,0.003829,1.062558,0.754035,0.063717,0.058875,0.153316
136,frozenset({Organic Cucumber}),frozenset({Organic Hass Avocado}),0.024181,0.066650,0.005441,0.225000,3.375850,1.0,0.003829,1.204323,0.721218,0.063717,0.169658,0.153316


In [21]:
def recommend_from_cart(cart_items, rules, top_n=5):
    cart_set = set(cart_items)

    matched_rules = rules[
        rules["antecedents"].apply(lambda x: set(x).issubset(cart_set))
    ].copy()

    if matched_rules.empty:
        return []

    matched_rules["antecedent_size"] = matched_rules["antecedents"].apply(len)

    matched_rules = matched_rules.sort_values(
        by=["antecedent_size", "lift", "confidence"],
        ascending=[False, False, False]
    )

    recommendations = []

    for _, row in matched_rules.iterrows():
        consequents = list(row["consequents"])
        for item in consequents:
            if item not in cart_set and item not in recommendations:
                recommendations.append(item)

    return recommendations[:top_n]




In [16]:
recommend_from_cart(["Limes"], rules)
recommend_from_cart(["Limes", "Organic Cilantro"], rules)
recommend_from_cart(["Organic Hass Avocado", "Organic Lemon"], rules)

['Organic Cucumber',
 'Organic Raspberries',
 'Organic Zucchini',
 'Organic Strawberries',
 'Organic Yellow Onion']

In [20]:
recommend_from_cart(["Limes", "Organic Cilantro"], rules)

['Large Lemon',
 'Organic Avocado',
 'Organic Hass Avocado',
 'Organic Baby Spinach',
 'Banana']

In [18]:
import pickle

with open("rules.pkl", "wb") as f:
    pickle.dump(rules, f)

In [19]:
with open("rules.pkl", "rb") as f:
    rules = pickle.load(f)

In [22]:
popular_products = (
    merged_df["product_name"]
    .value_counts()
    .head(20)
    .index
    .tolist()
)

popular_products

['Banana',
 'Bag of Organic Bananas',
 'Organic Strawberries',
 'Organic Baby Spinach',
 'Organic Hass Avocado',
 'Organic Avocado',
 'Large Lemon',
 'Strawberries',
 'Organic Raspberries',
 'Limes',
 'Organic Whole Milk',
 'Organic Yellow Onion',
 'Organic Garlic',
 'Organic Zucchini',
 'Cucumber Kirby',
 'Organic Blueberries',
 'Organic Lemon',
 'Organic Fuji Apple',
 'Honeycrisp Apple',
 'Apple Honeycrisp Organic']

In [23]:
import pickle

with open("popular.pkl", "wb") as f:
    pickle.dump(popular_products, f)

In [24]:
from mlxtend.frequent_patterns import fpgrowth

In [25]:
fp_itemsets = fpgrowth(
    basket_df,
    min_support=0.005,
    use_colnames=True
)

In [26]:
fp_itemsets.head()

,support,itemsets
0,0.022317,frozenset({Carrots})
1,0.020806,frozenset({Michigan Organic Kale})
2,0.073904,frozenset({Organic Baby Spinach})
3,0.015617,frozenset({Organic Ginger Root})
4,0.014156,frozenset({Unsweetened Almondmilk})


In [27]:
fp_itemsets.shape

(184, 2)

In [28]:
print("Apriori itemsets:", frequent_itemsets.shape)
print("FP-Growth itemsets:", fp_itemsets.shape)

Apriori itemsets: (184, 2)
FP-Growth itemsets: (184, 2)
